# Class 10 - Introduction to Pandas
### Module 3 . Week 4 . Friday

NumPy was excellent for numerical arrays. But real-world datasets contain
names, dates, missing values, mixed data types, and column labels. This is
why Pandas exists.

Today you'll learn how analysts load, inspect, and understand a dataset
before performing any analysis. We will load one real dataset - the Titanic
passenger manifest - and use it for everything in this notebook.

## Learning Objectives

By the end of today's notebook you should be able to:

- Create a `Series` and a `DataFrame` from scratch
- Reading data: `read_csv`, `read_excel`, `read_json`
- Run the standard first-look checklist on any new dataset (`head`, `tail`, `info`, `describe`)
- Access columns three different ways and know which to use when
- Sort a DataFrame and explain why the index doesn't reset

## Why Pandas?

NumPy arrays are fast but `homogeneous` - every element shares one dtype, and
there are no column names. A patient record needs a name (text), an age
(int), a temperature (float), and an admission flag (bool) - all in one row.

Pandas solves this by building two structures on top of NumPy:
- **Series** - a labelled 1D array (one column)
- **DataFrame** - a labelled 2D table composed of multiple series (many columns, possibly different dtypes)

Every Pandas column is, underneath, a NumPy array with a label attached.
Everything from Module 2 - vectorisation, boolean masking, broadcasting -
still applies. Pandas just adds structure on top.
This is the library you'll use for almost every real analytics task: loading
data, inspecting it, cleaning it, and reshaping it before analysis.

> Think of NumPy as the engine and Pandas as the vehicle. Pandas is built on top of NumPy and uses NumPy arrays internally while providing a much richer interface for working with real-world data.

In [ ]:
!pip install pandas -q

In [6]:
import pandas as pd
import numpy as np

DATA_DIR = "./datasets"   # change this if your folder structure differs bois!!!

print(f'Pandas Version : {pd.__version__}')
print(f'NumPy Version  : {np.__version__}')

# pd.set_option("display.max_columns", None)
# pd.set_option("display.width", 120)

Pandas Version : 2.2.1
NumPy Version  : 1.26.4


## 1. Pandas Data Structures

Two objects. Everything in Pandas is one of these.

### 1.1 Series - a labelled 1D array

A **Series** is the simplest data structure in Pandas.
Think of it as a single column in an Excel spreadsheet.

Each value has:
- an **index** (row label)
- a **value**
Internally, a Series stores both pieces of information together.

In [4]:
#the simplest way to build a Series is from a Python list

temps = pd.Series([36.5, 38.9, 37.1, 39.4])     #provide name='Temperature'
print(temps)
print(f"\ntype: {type(temps).__name__}   dtype: {temps.dtype}")
print(f"values: {temps.values}")          #the underlying NumPy array
print(f"index:  {temps.index}")

<IPython.core.display.Javascript object>

0    36.5
1    38.9
2    37.1
3    39.4
dtype: float64

type: Series   dtype: float64
values: [36.5 38.9 37.1 39.4]
index:  RangeIndex(start=0, stop=4, step=1)


#### 1.1a Creating a Series

Three common sources: a list, a dict, or a NumPy array.

In [37]:
# From a list - gets a default 0,1,2... index
from_list = pd.Series([7088, 2591, 567])

print(f"from_list:\n{from_list}")

from_list:
0    7088
1    2591
2     567
dtype: int64


In [38]:
# From a dict - its keys become the index and its values become the Series values
from_dict = pd.Series({"BRCA1": 7088, "TP53": 2591, "KRAS": 567})

print(f"\nfrom_dict:\n{from_dict}")


from_dict:
BRCA1    7088
TP53     2591
KRAS      567
dtype: int64


In [39]:
#from a NumPy array - connects directly back to Module 2
from_array = pd.Series(np.array([1.2, 0.8, 2.1]), index=["S1","S2","S3"])

print(f"\nfrom_array:\n{from_array}")


from_array:
S1    1.2
S2    0.8
S3    2.1
dtype: float64


#### 1.1b Creating a `Series with a Custom Index`
Often, row labels have meaningful names instead of numbers.

In [ ]:
temperatures = pd.Series(
    [36.5, 37.1, 38.4, 36.9],
    index=['Ali', 'Sara', 'Ahmed', 'Fatima'],       #or we can use patient ID as index=["P001", "P002", "P003", "P004"]
    name='Temperature'
)

temperatures

Ali       36.5
Sara      37.1
Ahmed     38.4
Fatima    36.9
Name: Temperature, dtype: float64

### 1.2 DataFrame - a labelled 2D table

- A **DataFrame** is a two-dimensional tabular data structure consisting of rows and columns. Think of it as an Excel worksheet or a SQL table or dict of Series sharing one index
- Each column is internally stored as a Pandas Series, allowing different columns to have different data types.

In [15]:
sample = pd.DataFrame({"name": ["Ahmad", "Sara"], "temp": [38.9, 36.7]})
print(f"\n{sample}")
print(f"type: {type(sample).__name__}")


    name  temp
0  Ahmad  38.9
1   Sara  36.7
type: DataFrame


### 1.2a Creating a DataFrame

The two patterns we'll use when building small reference tables or test data by hand.

**From dict of lists:**
- Every key becomes a column.
- Every list becomes one column of observations.
- All lists must have the same length.
- This approach is excellent for demonstrations, unit tests, and manually entering small datasets.

In [ ]:
# dict of lists - each key becomes a column
df_a = pd.DataFrame({
    "gene":   ["BRCA1", "TP53", "KRAS"],
    "chr":    ["17", "17", "12"],
    "length": [7088, 2591, 567],
})

print(f"dict of lists:\n{df_a}")

dict of lists:
    gene chr  length
0  BRCA1  17    7088
1   TP53  17    2591
2   KRAS  12     567


In [36]:
# Another example, Creating a DataFrame from a dict of lists most common manual creation method
patients = pd.DataFrame({
    "id":     ["P001", "P002", "P003", "P004", "P005"],
    "name":   ["Ahmad Raza", "Sara Khan", "Bilal Ahmed", "Zara Malik", "Hassan Ali"],
    "age":    [34, 28, 51, 45, 67],
    "temp":   [38.9, 36.7, 39.4, 37.1, 38.2],
    "bp_sys": [145, 118, 158, 132, 142],
})
print(patients)
print(f"\ntype: {type(patients).__name__}")
print(f"shape: {patients.shape}")

     id         name  age  temp  bp_sys
0  P001   Ahmad Raza   34  38.9     145
1  P002    Sara Khan   28  36.7     118
2  P003  Bilal Ahmed   51  39.4     158
3  P004   Zara Malik   45  37.1     132
4  P005   Hassan Ali   67  38.2     142

type: DataFrame
shape: (5, 5)


In [ ]:
# list of dicts - common when data arrives from an API/JSON response
df_b = pd.DataFrame([
    {"gene": "BRCA1", "chr": "17", "length": 7088},
    {"gene": "TP53",  "chr": "17", "length": 2591},
])

print(f"\nlist of dicts:\n{df_b}")

# np.dtype(df_b.length)


list of dicts:
    gene chr  length
0  BRCA1  17    7088
1   TP53  17    2591


dtype('int64')

In [22]:
print(f"\nSame structure: {list(df_a.columns) == list(df_b.columns)}")


Same structure: True


In [35]:
# DataFrame from a NumPy array - connects directly back to Module 2
matrix = np.array([[1.2, 0.8, 2.1],
                   [0.9, 1.5, 0.7],
                   [2.0, 1.8, 1.3]])
matrix_df = pd.DataFrame(matrix, columns=["BRCA1","TP53","KRAS"],
                         index=["Sample1","Sample2","Sample3"])
print(f"\n{matrix_df}")


         BRCA1  TP53  KRAS
Sample1    1.2   0.8   2.1
Sample2    0.9   1.5   0.7
Sample3    2.0   1.8   1.3


### Qs
1. What is the default index of a Series?
2. How does Pandas create a Series from a dictionary?
3. Why is a Series considered one-dimensional?
4. Which creation method would you use if your data already exists in a NumPy array?

### How to import any Data aside csv with any Separator

In [ ]:
# from io import StringIO

# # Common read_csv parameters demonstrated
# messy_csv = '''Patient ID;Name;Age;Temp
# P001;Ahmad;34;38.9
# P002;Sara;28;36.7
# P003;Bilal;NA;39.4
# '''

# # sep=';' for non-comma delimited files
# df = pd.read_csv(StringIO(messy_csv), sep=";", na_values=["NA"])
# print(df)
# print(f"\ndtypes:\n{df.dtypes}")

# # read_excel() and read_json() have nearly identical syntax:
# # df = pd.read_excel("data.xlsx", sheet_name="Sheet1")
# # df = pd.read_json("data.json")
# print("\nread_excel() and read_json() follow the same pattern as read_csv()")

<IPython.core.display.Javascript object>

  Patient ID   Name   Age  Temp
0       P001  Ahmad  34.0  38.9
1       P002   Sara  28.0  36.7
2       P003  Bilal   NaN  39.4

dtypes:
Patient ID     object
Name           object
Age           float64
Temp          float64
dtype: object

read_excel() and read_json() follow the same pattern as read_csv()


In [ ]:
# df = pd.read_csv(f"{DATA_DIR}/patients.csv")        #notice the index
# df = pd.read_csv(f"{DATA_DIR}/patients.csv", index_col = 'patient_id')      #notice the index

# print(f"Loaded {len(df)} passenger records")
# df.head()

Loaded 16 passenger records


,patient_id,name,age,gender,bp_sys,visit_date,temp
0,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,38.9
1,P002,Sara Khan,28 years,female,118.0,2024-01-11,36.7
2,P003,BILAL ahmed,51yo,M,NaN,2024-01-12,39.4
3,P001,ahmad RAZA,34 yrs,Male,145.0,2024-01-10,37.1
4,P004,Zara Malik,45 yr old,FEMALE,132.0,2024-01-13,38.2


## 2. Reading External Data

In professional data analysis, datasets are usually stored in external files rather than being manually typed. We load these. `pd.read_csv()` is the function we'll call most often in this course. `pd.read_excel()` and `pd.read_json()` follow the same pattern.

Pandas provides powerful functions to read data from many file formats including:
- CSV
- Excel
- JSON
- SQL Databases
- HTML Tables
- Parquet
- Feather
- APIs

In [12]:
# Load the Titanic passenger manifest - this is REAL historical data

# df = pd.read_csv(f"{DATA_DIR}/titanic.csv")
df = pd.read_csv(r'C:\Users\PMLS\Desktop\hands-on-data-analytics-python\notebooks\03_Module3_Pandas\week_04\datasets\titanic.csv')

print(f"Loaded {len(df)} passenger records")
df.head()

<IPython.core.display.Javascript object>

Loaded 891 passenger records


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. First Look at the Dataset

Whenever a new dataset is loaded, experienced analysts never begin analysis immediately.

Instead, they first inspect the data to understand:
- What variables exist?
- How many observations are present?
- Which columns contain missing values?
- Which columns are numerical?
- Which columns are categorical?
- Are there any unexpected values?

This exploratory step helps prevent mistakes later in the analysis.

Whenever you load a dataset, follow this sequence:
1. `head()`
2. `tail()`
3. `info()`
4. `describe()`
5. `shape`
6. `columns`
7. `dtypes`

Developing this habit will save considerable debugging time in future projects.

In [14]:
a = '''
- Are the column names meaningful?
- Do the values appear reasonable?
- Are there obvious missing values?
- Does the dataset appear to have loaded correctly?
'''

print(a)

df.head()       # first 5 rows

# df.head(3)      #first three rows


- Are the column names meaningful?
- Do the values appear reasonable?
- Are there obvious missing values?
- Does the dataset appear to have loaded correctly?



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [43]:
df.tail(3)      # last 3 rows - confirm the end looks consistent with the start

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [ ]:
print(f"shape:   {df.shape}")       #returns a tuple containing rows and columns
print(f"columns: {list(df.columns)}")
# print(f"dtypes:\n{df.dtypes}")  #will be same as info 

shape:   (891, 12)
columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [58]:
b = '''  
info() reports:
- Number of rows
- Number of columns
- Column names
- Non-null values
- Data types
- Memory usage
'''

df.info()       # schema, dtypes, non-null counts, memory - the single most useful command

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


**Think before moving on:**
- Which columns have missing values? *(check the "Non-Null Count" column above)*
- Which columns are numeric vs text/categorical?
- `Age` is stored as `float64` - why would a whole-number quantity like age need decimals? *(hint: missing values force the column to float)*
- Why is `Cabin` missing so many values?

#### **Descriptive Statistics**

| Statistic | Meaning |
|-----------|---------|
| count | Number of non-missing values |
| mean | Arithmetic average |
| std | Standard deviation |
| min | Smallest value |
| 25% | First quartile |
| 50% | Median |
| 75% | Third quartile |
| max | Largest value |

In [18]:
df.describe()   # statistical summary - numeric columns only by default
# .describe() with include='all' - works on non-numeric columns too

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


`.describe()` returns: `count` (non-null values), `mean`, `std` (spread), then the five-number summary - `min`, `25%`, `50%` (median), `75%`, `max`. Notice `Age` has a lower `count` than `PassengerId` - that gap is the missing values you just saw in `.info()`.

## 4. Memory Usage
Large datasets may consume hundreds of megabytes or even several gigabytes of memory.

Understanding memory usage helps us:

- identify expensive columns
- optimize storage
- improve performance
- reduce RAM consumption

Pandas provides the `memory_usage()` method to calculate the amount of memory consumed by each column.

Using `deep=True` allows Pandas to accurately estimate memory consumed by object (string) columns.

In [59]:
print(df.memory_usage(deep=True))
print(f"\nTotal: {df.memory_usage(deep=True).sum() / 1024:.1f} KB"
      f"(\nTotal: {df.memory_usage(deep=True).sum()} bytes)")

Index            132
PassengerId     7128
Survived        7128
Pclass          7128
Name           67685
Sex            47851
Age             7128
SibSp           7128
Parch           7128
Ticket         49674
Fare            7128
Cabin          32712
Embarked       44514
dtype: int64

Total: 285.6 KB(
Total: 292464 bytes)


In [60]:
heaviest = df.memory_usage(deep=True).drop("Index").idxmax()        #returns the column consuming the greatest amount of memory.
print(f'Heaviest column: "{heaviest}"  (text columns cost far more than numeric ones)')

Heaviest column: "Name"  (text columns cost far more than numeric ones)


- Which column consumes the most memory?
- Why do object (string) columns usually require more memory?
- How might categorical data reduce memory usage?
- Why is memory optimization important when working with very large datasets?

## 5. Column Selection

Three access patterns. Know all three - you'll see all three in other people's code.

In [ ]:
# Single bracket (works for any column) -> returns a Series (1D)
ages = df["Age"]
print(f"df['Age']        -> {type(ages).__name__}, shape {ages.shape}")

ages

df['Age']        -> Series, shape (891,)


0      22.0
1      38.0
2      26.0
3      35.0
4      35.0
       ... 
886    27.0
887    19.0
888     NaN
889    26.0
890    32.0
Name: Age, Length: 891, dtype: float64

In [62]:
# Attribute access -> identical to single bracket, but breaks on names with spaces/special characters
ages_attr = df.Age
print(f"df.Age           -> matches: {(ages.equals(ages_attr))}")

df.Age           -> matches: True


In [63]:
# Double bracket -> DataFrame (2D), even for one column
ages_df = df[["Age"]]
print(f"df[['Age']]      -> {type(ages_df).__name__}, shape {ages_df.shape}")

df[['Age']]      -> DataFrame, shape (891, 1)


In [ ]:
# Multiple columns
subset = df[["Name", "Age", "Fare"]]
print(f"\n{subset.head(3)}")

# subset


                                                Name   Age     Fare
0                            Braund, Mr. Owen Harris  22.0   7.2500
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  38.0  71.2833
2                             Heikkinen, Miss. Laina  26.0   7.9250


,Name,Age,Fare
0,"Braund, Mr. Owen Harris",22.0,7.2500
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
2,"Heikkinen, Miss. Laina",26.0,7.9250
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000
4,"Allen, Mr. William Henry",35.0,8.0500
...,...,...,...
886,"Montvila, Rev. Juozas",27.0,13.0000
887,"Graham, Miss. Margaret Edith",19.0,30.0000
888,"Johnston, Miss. Catherine Helen ""Carrie""",NaN,23.4500
889,"Behr, Mr. Karl Howell",26.0,30.0000


| Expression | Returns |
|------------|---------|
| `df['Age']` | Series |
| `df[['Age']]` | DataFrame |
| `df[['Age','Fare']]` | DataFrame |

Understanding this distinction is important because some Pandas methods behave differently depending on whether the input is a Series or a DataFrame.

## Sorting

- The `sort_index()` method orders rows according to the DataFrame index rather than column values.
- The `sort_values(by = col or list of columns , ascending = bool/list of boolean , inplace = bool)` method orders rows according to the `specified` column values.


In [51]:
df_copy = df.copy()

df_copy.sort_index(axis = 1)        #axis = 0 sort on the basis of rows, while axis = 1 sorts columns alphabetically

,Age,Cabin,Embarked,Fare,Name,Parch,PassengerId,Pclass,Sex,SibSp,Survived,Ticket
0,22.0,NaN,S,7.2500,"Braund, Mr. Owen Harris",0,1,3,male,1,0,A/5 21171
1,38.0,C85,C,71.2833,"Cumings, Mrs. John Bradley (Florence Briggs Th...",0,2,1,female,1,1,PC 17599
2,26.0,NaN,S,7.9250,"Heikkinen, Miss. Laina",0,3,3,female,0,1,STON/O2. 3101282
3,35.0,C123,S,53.1000,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",0,4,1,female,1,1,113803
4,35.0,NaN,S,8.0500,"Allen, Mr. William Henry",0,5,3,male,0,0,373450
...,...,...,...,...,...,...,...,...,...,...,...,...
886,27.0,NaN,S,13.0000,"Montvila, Rev. Juozas",0,887,2,male,0,0,211536
887,19.0,B42,S,30.0000,"Graham, Miss. Margaret Edith",0,888,1,female,0,1,112053
888,NaN,NaN,S,23.4500,"Johnston, Miss. Catherine Helen ""Carrie""",2,889,3,female,1,0,W./C. 6607
889,26.0,C148,C,30.0000,"Behr, Mr. Karl Howell",0,890,1,male,0,1,111369


In [ ]:
df.sort_values(by="Fare", ascending=False).head()       #always sort to rows axis = 0

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
258,259,1,1,"Ward, Miss. Anna",female,35.0,0,0,PC 17755,512.3292,NaN,C
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C
679,680,1,1,"Cardeza, Mr. Thomas Drake Martinez",male,36.0,0,1,PC 17755,512.3292,B51 B53 B55,C
88,89,1,1,"Fortune, Miss. Mabel Helen",female,23.0,3,2,19950,263.0000,C23 C25 C27,S
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.0000,C23 C25 C27,S


In [ ]:
# sort_values does NOT modify df in place (# unless you pass inplace=True), and does NOT reset the index
top_fares = df.sort_values(by="Fare", ascending=False)
print(top_fares.head(3)[["Name","Fare"]])
print(f"\nNotice the index column on the left - it's NOT 0,1,2...")
print(f"Those are the ORIGINAL row positions before sorting.")
print(f"\ndf itself is still unsorted: {df['Fare'].head(3).tolist()}")

                                   Name      Fare
258                    Ward, Miss. Anna  512.3292
737              Lesurer, Mr. Gustave J  512.3292
679  Cardeza, Mr. Thomas Drake Martinez  512.3292

Notice the index column on the left - it's NOT 0,1,2...
Those are the ORIGINAL row positions before sorting.

df itself is still unsorted: [7.25, 71.2833, 7.925]


`sort_values()`: Sorts the rows based on the data inside one or more specified columns.

        - to find extremes in data
        - for multi-level data sorting like `by = ['Class','Age']`, sort by Class first, and then by Age within each class.
  
`sort_index()`: Sorts the rows (or columns) based on their labels (the index).

        - With Time-Series data, if index have time stamps/dates to ensure your timeline is in perfect chronological order.
        - Restores order after operations like `groupby` or `manual merging`.

In [22]:
a = df.sort_values(by=['Pclass', 'Age'], ascending=[True, False])
a.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
630,631,1,1,"Barkworth, Mr. Algernon Henry Wilson",male,80.0,0,0,27042,30.0000,A23,S
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C
493,494,0,1,"Artagaveytia, Mr. Ramon",male,71.0,0,0,PC 17609,49.5042,NaN,C
745,746,0,1,"Crosby, Capt. Edward Gifford",male,70.0,1,1,WE/P 5735,71.0000,B22,S
54,55,0,1,"Ostby, Mr. Engelhart Cornelius",male,65.0,0,1,113509,61.9792,B30,C


In [25]:
a.tail(150)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
16,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.1250,NaN,Q
119,120,0,3,"Andersson, Miss. Ellis Anna Maria",female,2.0,4,2,347082,31.2750,NaN,S
205,206,0,3,"Strom, Miss. Telma Matilda",female,2.0,0,1,347054,10.4625,G6,S
479,480,1,3,"Hirvonen, Miss. Hildur E",female,2.0,0,1,3101298,12.2875,NaN,S
642,643,0,3,"Skoog, Miss. Margit Elizabeth",female,2.0,3,2,347088,27.9000,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
859,860,0,3,"Razi, Mr. Raihed",male,NaN,0,0,2629,7.2292,NaN,C
863,864,0,3,"Sage, Miss. Dorothy Edith ""Dolly""",female,NaN,8,2,CA. 2343,69.5500,NaN,S
868,869,0,3,"van Melkebeke, Mr. Philemon",male,NaN,0,0,345777,9.5000,NaN,S
878,879,0,3,"Laleff, Mr. Kristo",male,NaN,0,0,349217,7.8958,NaN,S


## Common Mistakes

New Pandas users often encounter the following issues:

1. Forgetting the double brackets when selecting multiple columns.
2. Using attribute access (`df.column`) for columns containing spaces.
3. Forgetting that most Pandas methods return a new DataFrame instead of modifying the original.
4. Performing analysis before inspecting the dataset.
5. Ignoring missing values shown by `info()`.

## Mini Challenge

*How many passengers have a missing `Cabin` value? You saw this number already in `.info()`. What was it?*

Now verify your answer:

In [52]:
missing_cabin = df["Cabin"].isnull().sum()
print(f"Missing Cabin values: {missing_cabin} out of {len(df)} ({missing_cabin/len(df)*100:.0f}%)")

Missing Cabin values: 687 out of 891 (77%)


## Practical

*All four practical use the same `df` already loaded above - no new data.*

### Practical 1 - Series vs DataFrame: a one-character bug

In [53]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin',
       'Embarked'],
      dtype='object')

In [54]:
age_series = df["Age"]          # single bracket -> Series
age_dataframe = df[["Age", "Fare"]]     # double bracket -> DataFrame

print(f"age_series type:    {type(age_series).__name__}")
print(f"age_dataframe type: {type(age_dataframe).__name__}")


print(f"\nSeries.mean() returns a single number: {age_series.mean():.2f}")
print(f"DataFrame.mean() returns a Series:\n{age_dataframe.mean()}")
print()
print("Lesson: [col] gives 1D Series. [[col]] gives 2D DataFrame with index as column name. Know which you have.")

age_series type:    Series
age_dataframe type: DataFrame

Series.mean() returns a single number: 29.70
DataFrame.mean() returns a Series:
Age     29.699118
Fare    32.204208
dtype: float64

Lesson: [col] gives 1D Series. [[col]] gives 2D DataFrame with index as column name. Know which you have.


### Practical 2 - .info() catches a dtype problem before it costs you
**Context:** Imagine `Age` had been loaded as text because of one stray value like `"unknown"`. Nothing crashes on load - until you try to compute statistics.

In [50]:
# Simulate the failure mode on a COPY - illustrating what .info() would catch
broken = df[["Age"]].astype(object).copy()
broken.loc[0, "Age"] = "unknown"     # plant one bad value

print(f"dtype after one bad value: {broken['Age'].dtype}   <- should be float64!")

try:
    print(f"Mean age: {broken['Age'].mean()}")
except Exception as e:
    print(f"ERROR: {type(e).__name__}: {e}")

print()
print("ALWAYS run .info() right after loading. An expected-numeric column")
print("showing dtype='object' is a red flag - We fix this properly with pd.to_numeric() in Class 12.")

dtype after one bad value: object   <- should be float64!
ERROR: TypeError: can only concatenate str (not "float") to str

ALWAYS run .info() right after loading. An expected-numeric column
showing dtype='object' is a red flag - We fix this properly with pd.to_numeric() in Class 12.


### Practical 3 - sort_values doesn't reset the index - and that's a feature
**Context:** After sorting, `.iloc[0]` and `.loc[0]` give two different rows. Why?

In [55]:
sorted_df = df.sort_values(by="Fare", ascending=False)

print(f"sorted_df.iloc[0]   (position 0 - the highest fare):")
print(sorted_df.iloc[0][["Name","Fare"]])

print(f"\nsorted_df.loc[0]    (LABEL 0 - the ORIGINAL first row, Mr. Owen Harris):")
print(sorted_df.loc[0][["Name","Fare"]])

print()
print("iloc = position-based (always 0,1,2... regardless of sort order)")
print("loc  = label-based (refers to the ORIGINAL index labels)")
print("Fix: .sort_values('Fare').reset_index(drop=True) if you want fresh labels")

sorted_df.iloc[0]   (position 0 - the highest fare):
Name    Ward, Miss. Anna
Fare            512.3292
Name: 258, dtype: object

sorted_df.loc[0]    (LABEL 0 - the ORIGINAL first row, Mr. Owen Harris):
Name    Braund, Mr. Owen Harris
Fare                       7.25
Name: 0, dtype: object

iloc = position-based (always 0,1,2... regardless of sort order)
loc  = label-based (refers to the ORIGINAL index labels)
Fix: .sort_values('Fare').reset_index(drop=True) if you want fresh labels


### Practical 4 - memory_usage(deep=True): why text columns lie about their size
A dataset 'feels' small but uses way more RAM than expected. The culprit is almost always string columns.

In [52]:
print("Shallow (default - misleading for text columns):")
print(df.memory_usage(deep=False))
print(f"Total (shallow): {df.memory_usage(deep=False).sum()/1024:.1f} KB")

print("\nDeep (accurate - accounts for actual string content):")
print(df.memory_usage(deep=True))
print(f"Total (deep): {df.memory_usage(deep=True).sum()/1024:.1f} KB")

print()
print("Object (text) columns store POINTERS in the shallow view.")
print("Always use deep=True before deciding if a dataset fits in RAM.")

Shallow (default - misleading for text columns):
Index           132
PassengerId    7128
Survived       7128
Pclass         7128
Name           7128
Sex            7128
Age            7128
SibSp          7128
Parch          7128
Ticket         7128
Fare           7128
Cabin          7128
Embarked       7128
dtype: int64
Total (shallow): 83.7 KB

Deep (accurate - accounts for actual string content):
Index            132
PassengerId     7128
Survived        7128
Pclass          7128
Name           74813
Sex            54979
Age             7128
SibSp           7128
Parch           7128
Ticket         56802
Fare            7128
Cabin          34344
Embarked       51626
dtype: int64
Total (deep): 315.0 KB

Object (text) columns store POINTERS in the shallow view.
Always use deep=True before deciding if a dataset fits in RAM.


## Practical Exercise - Titanic Dataset First Look
**Difficulty: Low**

Use the `df` already loaded above. Do not reload or recreate anything - answer using only Pandas, no loops.

In [53]:
total_passengers = len(df)
avg_fare         = df["Fare"].mean()
unique_ports     = df["Embarked"].dropna().unique()
pct_male         = (df["Sex"] == "male").mean() * 100
age_min, age_max = df["Age"].min(), df["Age"].max()

summary = pd.DataFrame({
    "Metric": ["Total Passengers", "Average Fare", "Unique Embarkation Ports",
              "% Male", "Min Age", "Max Age"],
    "Value": [total_passengers, round(avg_fare,2), ", ".join(unique_ports),
             round(pct_male,1), age_min, age_max]
})
summary

,Metric,Value
0,Total Passengers,891
1,Average Fare,32.2
2,Unique Embarkation Ports,"S, C, Q"
3,% Male,64.8
4,Min Age,0.42
5,Max Age,80.0


In [54]:
# Memory report
mem = df.memory_usage(deep=True)
print(f"Total memory: {mem.sum()/1024:.1f} KB")
print(f"Largest column: {mem.drop('Index').idxmax()} ({mem.drop('Index').max()/1024:.1f} KB)")

# Bonus - survival rate, a preview of groupby in Class 13
print(f"\nOverall survival rate: {df['Survived'].mean()*100:.1f}%")

Total memory: 315.0 KB
Largest column: Name (73.1 KB)

Overall survival rate: 38.4%


In [55]:
#Bonus: survival rate by class — preview of what's coming in groupby (Class 13)
print(f"\nSurvival rate overall: {df['Survived'].mean()*100:.1f}%")
print(f"\nSorted by fare (highest first):")
print(df.sort_values(by="Fare", ascending=False)[["Name","Fare","Survived"]].head())


Survival rate overall: 38.4%

Sorted by fare (highest first):
                                   Name      Fare  Survived
258                    Ward, Miss. Anna  512.3292         1
737              Lesurer, Mr. Gustave J  512.3292         1
679  Cardeza, Mr. Thomas Drake Martinez  512.3292         1
88           Fortune, Miss. Mabel Helen  263.0000         1
27       Fortune, Mr. Charles Alexander  263.0000         0


## Summary

| Concept | Key point |
|---|---|
| Series | Labelled 1D array - a single column |
| DataFrame | 2D table - dict of Series sharing one index |
| `read_csv()` | The function you'll call most - load once, reuse the result |
| `.info()` | Run immediately after loading - catches dtype problems early |
| `.describe()` | Count, mean, std, then the five-number summary |
| `df['col']` vs `df[['col']]` | Single bracket -> Series; double bracket -> DataFrame |
| `sort_values()` | Doesn't reset the index - original labels travel with sorted rows |

## Homework

A dataset has a column `'Salary'` stored as the string `'$45,000'`. Predict, in
writing, why `df['Salary'].mean()` would fail right now - and sketch (in
words, not code) how you'd fix it. We solve exactly this in Class 12.

** What do you predict `df[df['age'] > 30]` does? We haven't covered boolean filtering on DataFrames yet - but you already know boolean masking from NumPy. Think about how it should translate.


Answer the following questions without running additional code.

1. What is the difference between a Series and a DataFrame?
2. Which DataFrame creation method is commonly used with REST APIs?
3. Why should we inspect a dataset before performing analysis?
4. What is the purpose of `deep=True` in `memory_usage()`?
5. What is the difference between `sort_values()` and `sort_index()`?
6. What is returned by `df[['Age']]`?
7. What is returned by `df['Age']`?